# Tiny gate training
This deliberately uses synthetic/replay-shaped values. It is an artifact exercise, not a trading result.

In [ ]:
import torch
from torch import nn

torch.manual_seed(7)
x = torch.randn(160, 300)  # 160 causal 30 x 10 windows
utility = torch.stack((x[:, -5], x[:, -4] * .6, -x[:, -1]), dim=1)
uniform = utility.mean(dim=0).mean()
static = (utility * torch.tensor([.5, .35, .15])).sum(1).mean()
print({'uniform': float(uniform), 'static': float(static)})

In [ ]:
model = nn.Sequential(nn.Linear(300,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,3))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for _ in range(40):
    w = torch.softmax(model(x), dim=1)
    loss = -(w * utility).sum(1).mean()
    opt.zero_grad(); loss.backward(); opt.step()
neural = (torch.softmax(model(x), dim=1) * utility).sum(1).mean()
print({'neural': float(neural), 'note': 'compare on chronological holdout before any promotion'})

In [ ]:
from pathlib import Path
import numpy as np

artifact = Path('../models/gate-demo.npz')
artifact.parent.mkdir(exist_ok=True)
first, second, third = (layer for layer in model if isinstance(layer, nn.Linear))
np.savez(artifact, schema_version=np.array('gate-npz-v1'), w1=first.weight.detach().numpy().T, b1=first.bias.detach().numpy(), w2=second.weight.detach().numpy().T, b2=second.bias.detach().numpy(), w3=third.weight.detach().numpy().T, b3=third.bias.detach().numpy())
from market_gate.gate import load_numpy_gate
runtime_gate = load_numpy_gate(artifact)
assert abs(sum(runtime_gate.predict(x[0].reshape(30, 10).numpy()).values()) - 1) < 1e-6
print('NPZ runtime artifact save/load verified')